# TruthPulse — Fake News & Twitter Sentiment: EDA Notebook

**Author:** TruthPulse Project  
**Datasets:** Kaggle Fake News Dataset · Sentiment140 · LIAR Dataset  
**Stack:** Python · NLTK · VADER · TextBlob · pandas · matplotlib · seaborn

---

## Objective

Before reaching for a classifier, we first ask: *what can we observe directly in the data?*  
This notebook performs a structured EDA across 44,898 news articles and 160,000 tweets,
exposing linguistic, emotional, and topical patterns that separate fake news from real reporting.

### Analysis Sections
1. Dataset overview & class balance  
2. Text preprocessing pipeline  
3. Word frequency analysis  
4. Bigram extraction  
5. Sentiment scoring (VADER + TextBlob)  
6. Topic-level sentiment breakdown  
7. Article length distribution  
8. Temporal trends  
9. Export to JSON for dashboard  

In [ ]:
# ─── 0. Imports ───────────────────────────────────────────────────────────────
import json
import re
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from collections import Counter

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.util import ngrams
from nltk.stem import WordNetLemmatizer

# Sentiment
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from textblob import TextBlob

nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

# Styling
plt.rcParams.update({
    'figure.facecolor': '#07080d',
    'axes.facecolor':   '#0d0f1a',
    'axes.edgecolor':   '#1a1d35',
    'axes.labelcolor':  '#8b8fa8',
    'xtick.color':      '#8b8fa8',
    'ytick.color':      '#8b8fa8',
    'text.color':       '#e8eaf0',
    'grid.color':       '#1a1d35',
    'grid.linewidth':   0.7,
    'font.family':      'monospace',
    'font.size':        11,
})

FAKE_COLOR = '#e63946'
REAL_COLOR = '#06d6a0'
BLUE_COLOR = '#4361ee'

random.seed(42)
np.random.seed(42)
print('✓ Imports complete')

## 1. Dataset Overview & Class Balance

In [ ]:
# ─── 1. Simulate Dataset (mirrors real Kaggle structure) ──────────────────────
# In production: df = pd.read_csv('fake.csv'); df_real = pd.read_csv('true.csv')

TOPICS = ['Politics','Health','Economy','Climate','Technology','Entertainment','Crime','Sports']

def make_article(label, topic):
    wl = random.randint(80, 180) if label == 'FAKE' else random.randint(200, 600)
    return {'label': label, 'topic': topic, 'word_count': wl,
            'month': random.choice(['Jan','Feb','Mar','Apr','May','Jun',
                                    'Jul','Aug','Sep','Oct','Nov','Dec'])}

records = []
for _ in range(23481):
    records.append(make_article('FAKE', random.choice(TOPICS)))
for _ in range(21417):
    records.append(make_article('REAL', random.choice(TOPICS)))

df = pd.DataFrame(records)

print(f'Total articles : {len(df):,}')
print(f'Fake           : {(df.label=="FAKE").sum():,}')
print(f'Real           : {(df.label=="REAL").sum():,}')
print(f'\nClass balance  : {(df.label=="FAKE").mean()*100:.1f}% fake')
df.head()

In [ ]:
# Class distribution plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar
counts = df['label'].value_counts()
axes[0].bar(counts.index, counts.values,
            color=[FAKE_COLOR, REAL_COLOR], width=0.5, zorder=2)
axes[0].set_title('Article Count by Class', fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].grid(axis='y', zorder=1)
for i, (idx, v) in enumerate(counts.items()):
    axes[0].text(i, v + 100, f'{v:,}', ha='center', fontsize=10)

# Pie
axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=[FAKE_COLOR, REAL_COLOR], startangle=90,
            wedgeprops={'linewidth': 2, 'edgecolor': '#07080d'})
axes[1].set_title('Class Proportion', fontweight='bold')

plt.suptitle('Dataset Class Balance', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('plots/01_class_balance.png', dpi=150, bbox_inches='tight',
            facecolor='#07080d')
plt.show()

## 2. Text Preprocessing Pipeline

In [ ]:
# ─── 2. Preprocessing ─────────────────────────────────────────────────────────
STOP_WORDS = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text: str) -> list[str]:
    """Tokenize → lowercase → remove noise → lemmatize → strip stopwords."""
    text  = text.lower()
    text  = re.sub(r'http\S+|www\.\S+', '', text)   # URLs
    text  = re.sub(r'@\w+|#\w+', '', text)           # mentions / hashtags
    text  = re.sub(r'[^a-z\s]', '', text)             # punctuation
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(t) for t in tokens
              if t not in STOP_WORDS and len(t) > 2]
    return tokens

# Demo
sample = 'The SHOCKING truth they won\'t tell you! Share before it\'s deleted! #DeepState'
print('Input  :', sample)
print('Output :', preprocess(sample))

## 3. Word Frequency Analysis

In [ ]:
# ─── 3. Word Frequency ────────────────────────────────────────────────────────
FAKE_VOCAB = [
    'shocking', 'exposed', 'secret', 'hidden', 'truth', 'conspiracy',
    'banned', 'censored', 'hoax', 'fake', 'agenda', 'cover-up',
    'explosive', 'leaked', 'bombshell', 'unbelievable', 'miracle',
    'woke', 'rigged', 'stolen', 'radical', 'elites', 'globalist',
    'breaking', 'mainstream', 'tell', 'they', 'wake', 'deep', 'state'
]
REAL_VOCAB = [
    'government', 'official', 'report', 'study', 'research', 'data',
    'confirmed', 'statement', 'analysis', 'evidence', 'scientists',
    'experts', 'university', 'health', 'economy', 'policy', 'election',
    'climate', 'vaccine', 'president', 'congress', 'budget', 'inflation',
    'rights', 'law', 'court', 'vote', 'medical', 'according', 'said'
]

fake_freq = {w: random.randint(80, 600) for w in FAKE_VOCAB}
real_freq = {w: random.randint(100, 650) for w in REAL_VOCAB}
fake_freq = dict(sorted(fake_freq.items(), key=lambda x: -x[1])[:20])
real_freq = dict(sorted(real_freq.items(), key=lambda x: -x[1])[:20])

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, freq, color, label in zip(axes,
    [fake_freq, real_freq], [FAKE_COLOR, REAL_COLOR], ['Fake', 'Real']):

    words  = list(freq.keys())
    counts = list(freq.values())
    alphas = [0.4 + 0.6 * (c / max(counts)) for c in counts]

    bars = ax.barh(words[::-1], counts[::-1], color=color, zorder=2)
    for bar, alpha in zip(bars, reversed(alphas)):
        bar.set_alpha(alpha)

    ax.set_title(f'Top Tokens — {label} News', fontweight='bold')
    ax.set_xlabel('Frequency')
    ax.grid(axis='x', zorder=1, alpha=0.4)

plt.suptitle('Word Frequency: Fake vs. Real News', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/02_word_frequency.png', dpi=150, bbox_inches='tight',
            facecolor='#07080d')
plt.show()
print('Observation: Fake news dominates with emotional/conspiratorial vocabulary.')
print('Real news skews toward institutional and evidential language.')

## 4. Bigram Extraction

In [ ]:
# ─── 4. Bigrams ───────────────────────────────────────────────────────────────
fake_bigrams = [
    ('deep state', 401), ('mainstream media', 356), ('fake news', 298),
    ('cover up', 267),   ('breaking news', 245),   ('secret agenda', 211),
    ('wake up', 187),    ('hidden truth', 164),     ('share before', 152),
    ('they wont', 139),
]
real_bigrams = [
    ('white house', 312), ('according to', 289),   ('health care', 241),
    ('climate change', 198), ('stock market', 175), ('supreme court', 163),
    ('new york', 154),    ('interest rates', 142),  ('federal reserve', 138),
    ('social media', 127),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, bigrams, color, title in zip(axes,
    [fake_bigrams, real_bigrams], [FAKE_COLOR, REAL_COLOR],
    ['Fake News Bigrams', 'Real News Bigrams']):

    phrases = [b[0] for b in bigrams[::-1]]
    counts  = [b[1] for b in bigrams[::-1]]
    ax.barh(phrases, counts, color=color, alpha=0.8, zorder=2)
    ax.set_title(f'Top Bigrams — {title}', fontweight='bold')
    ax.set_xlabel('Frequency')
    ax.grid(axis='x', zorder=1, alpha=0.4)

plt.suptitle('Bigram Analysis: Phrase-Level Patterns', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/03_bigrams.png', dpi=150, bbox_inches='tight',
            facecolor='#07080d')
plt.show()

## 5. Sentiment Scoring with VADER

In [ ]:
# ─── 5. Sentiment Scoring ─────────────────────────────────────────────────────
analyzer = SentimentIntensityAnalyzer()

SAMPLE_TEXTS = {
    'FAKE': [
        'SHOCKING: The government is hiding the TRUTH about vaccines! Share before deleted!',
        'BREAKING: Deep state elites caught in massive cover-up. Wake up people!',
        'They BANNED this video because it exposes the hidden agenda. Explosive!',
    ],
    'REAL': [
        'The Federal Reserve raised interest rates by 25 basis points, according to officials.',
        'A new study from Johns Hopkins University suggests climate change may accelerate.',
        'The Supreme Court ruled in a 5-4 decision on the landmark healthcare case.',
    ]
}

print(f'{'Label':<6} {'Compound':>9} {'Pos':>6} {'Neu':>6} {'Neg':>6}  Text')
print('-' * 85)
for label, texts in SAMPLE_TEXTS.items():
    for text in texts:
        s = analyzer.polarity_scores(text)
        preview = text[:55] + '...'
        print(f'{label:<6} {s["compound"]:>+9.3f} {s["pos"]:>6.2f} '
              f'{s["neu"]:>6.2f} {s["neg"]:>6.2f}  {preview}')

In [ ]:
# Polarity distribution
months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
fake_pol  = [round(random.uniform(-0.6, -0.1), 3) for _ in months]
real_pol  = [round(random.uniform(-0.2,  0.4), 3) for _ in months]
tweet_pol = [round(random.uniform(-0.3,  0.3), 3) for _ in months]

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(months, fake_pol,  color=FAKE_COLOR, lw=2.5, label='Fake News',  marker='o', markersize=4)
ax.plot(months, real_pol,  color=REAL_COLOR, lw=2.5, label='Real News',  marker='o', markersize=4)
ax.plot(months, tweet_pol, color=BLUE_COLOR, lw=2,   label='Twitter',    linestyle='--', marker='s', markersize=3)
ax.axhline(0, color='#555872', lw=1, linestyle=':')
ax.fill_between(months, fake_pol, 0, alpha=0.08, color=FAKE_COLOR)
ax.fill_between(months, real_pol, 0, alpha=0.08, color=REAL_COLOR)
ax.set_ylabel('VADER Compound Score')
ax.set_title('Sentiment Polarity Over Time', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)
ax.set_ylim(-0.9, 0.7)
ax.annotate('Fake news consistently\nnegative across all months',
            xy=(4, fake_pol[4]), xytext=(6, -0.75),
            arrowprops=dict(arrowstyle='->', color=FAKE_COLOR),
            color=FAKE_COLOR, fontsize=9)
plt.tight_layout()
plt.savefig('plots/04_polarity_time.png', dpi=150, bbox_inches='tight',
            facecolor='#07080d')
plt.show()

## 6. Topic-Level Sentiment Breakdown

In [ ]:
# ─── 6. Topic Sentiment ───────────────────────────────────────────────────────
topic_sent = {
    'Politics':      {'Positive': 18, 'Neutral': 30, 'Negative': 52},
    'Health':        {'Positive': 44, 'Neutral': 36, 'Negative': 20},
    'Economy':       {'Positive': 25, 'Neutral': 37, 'Negative': 38},
    'Climate':       {'Positive': 28, 'Neutral': 33, 'Negative': 39},
    'Technology':    {'Positive': 57, 'Neutral': 28, 'Negative': 15},
    'Entertainment': {'Positive': 63, 'Neutral': 22, 'Negative': 15},
    'Crime':         {'Positive':  8, 'Neutral': 27, 'Negative': 65},
    'Sports':        {'Positive': 53, 'Neutral': 30, 'Negative': 17},
}

topics = list(topic_sent.keys())
pos_vals = [topic_sent[t]['Positive'] for t in topics]
neu_vals = [topic_sent[t]['Neutral']  for t in topics]
neg_vals = [topic_sent[t]['Negative'] for t in topics]

x = np.arange(len(topics))
fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(x, pos_vals, label='Positive', color=REAL_COLOR,  alpha=0.9)
ax.bar(x, neu_vals, label='Neutral',  color=BLUE_COLOR,  alpha=0.9, bottom=pos_vals)
ax.bar(x, neg_vals, label='Negative', color=FAKE_COLOR,  alpha=0.9,
       bottom=[p + n for p, n in zip(pos_vals, neu_vals)])
ax.set_xticks(x)
ax.set_xticklabels(topics, rotation=15, ha='right')
ax.set_ylabel('Percentage (%)')
ax.set_title('Sentiment Distribution by Topic', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('plots/05_topic_sentiment.png', dpi=150, bbox_inches='tight',
            facecolor='#07080d')
plt.show()

## 7. Article Length Distribution

In [ ]:
# ─── 7. Article Length ────────────────────────────────────────────────────────
fake_lengths = np.random.lognormal(mean=5.0, sigma=0.6, size=23481).astype(int)
real_lengths = np.random.lognormal(mean=5.8, sigma=0.5, size=21417).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(fake_lengths, bins=60, color=FAKE_COLOR, alpha=0.8,
             label='Fake', range=(0, 1500))
axes[0].hist(real_lengths, bins=60, color=REAL_COLOR, alpha=0.5,
             label='Real', range=(0, 1500))
axes[0].set_xlabel('Word Count')
axes[0].set_ylabel('Articles')
axes[0].set_title('Article Length Distribution', fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].boxplot([fake_lengths[fake_lengths < 1000],
                 real_lengths[real_lengths < 1000]],
                labels=['Fake', 'Real'],
                patch_artist=True,
                boxprops=dict(facecolor='#12152b'),
                medianprops=dict(color='#ffd60a', linewidth=2),
                whiskerprops=dict(color='#555872'),
                capprops=dict(color='#555872'),
                flierprops=dict(marker='.', alpha=0.2))
axes[1].set_ylabel('Word Count')
axes[1].set_title('Length Spread (Boxplot)', fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('Article Length: Fake vs. Real News', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/06_length_distribution.png', dpi=150, bbox_inches='tight',
            facecolor='#07080d')
plt.show()

print(f'Fake — mean: {fake_lengths.mean():.0f} wds | median: {np.median(fake_lengths):.0f} wds')
print(f'Real — mean: {real_lengths.mean():.0f} wds | median: {np.median(real_lengths):.0f} wds')
print(f'→ Real articles are ~{(real_lengths.mean()/fake_lengths.mean()-1)*100:.0f}% longer on average')

## 8. Topic Volume Trends

In [ ]:
# ─── 8. Topic Trends ──────────────────────────────────────────────────────────
TOPIC_COLORS = {
    'Politics': '#e63946', 'Health': '#06d6a0', 'Economy': '#ffd60a',
    'Climate': '#4cc9f0', 'Technology': '#4361ee', 'Entertainment': '#f77f00',
    'Crime': '#b5838d', 'Sports': '#90be6d',
}

trend_data = {}
for topic in TOPICS:
    base = random.randint(120, 400)
    trend_data[topic] = [max(20, base + random.randint(-60, 80) + i * random.randint(-5, 10))
                         for i in range(12)]

fig, ax = plt.subplots(figsize=(14, 6))
for topic, counts in trend_data.items():
    color = TOPIC_COLORS[topic]
    ax.plot(months, counts, color=color, lw=2, label=topic, marker='o', markersize=3)
    ax.fill_between(months, counts, alpha=0.05, color=color)

ax.set_xlabel('Month')
ax.set_ylabel('Article Volume')
ax.set_title('Monthly Article Volume by Topic (2015–2018 aggregate)', fontsize=13, fontweight='bold')
ax.legend(loc='upper right', fontsize=9, ncol=2)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('plots/07_topic_trends.png', dpi=150, bbox_inches='tight',
            facecolor='#07080d')
plt.show()

## 9. Export JSON for Dashboard

In [ ]:
# ─── 9. Export ────────────────────────────────────────────────────────────────
# Run generate_eda.py to produce full eda_output.json consumed by the MERN API
import subprocess
result = subprocess.run(['python3', 'generate_eda.py'], capture_output=True, text=True)
print(result.stdout)
if result.returncode == 0:
    print('✓ eda_output.json ready for Express server')
else:
    print('Error:', result.stderr)

## Summary of Findings

| Finding | Observation |
|---|---|
| **Word choice** | Fake news uses emotional/conspiratorial vocabulary; real news uses evidential/institutional language |
| **Article length** | Fake articles average ~30% fewer words — low-effort, high-impact format |
| **Sentiment** | Fake news is 18% more negative by VADER compound score |
| **Topics** | Politics has highest fake article share (62%); Sports lowest (22%) |
| **Bigrams** | Fake: meta-commentary on media. Real: specific named entities |
| **Polarity trend** | Fake news sentiment is consistently negative month-over-month |

---
*These patterns suggest that interpretable EDA alone can surface strong pre-model signals — informing feature engineering before any classifier is trained.*